# Step 33 — Association coverage and diagnostic bar charts

Run after step 32. The notebook contains its plotting implementation directly, with shared date/save helpers. It reads saved CSVs and writes four full-width daily bar charts plus supporting daily CSVs. It does not reassign or validate picks, and does not plot waveforms with pick markers yet.

## Reading the figures

| Output | Meaning and denominator |
|---|---|
| `33_waveform_coverage.png` | Stacked WAV+PICKS and WAV_ONLY counts. Their sum should equal the waveform-index count for each recorded start day; that total is not stacked again. |
| `33_event_types.png` | Stacked counts of all three mutually exclusive container classes. These are association containers, not independent earthquakes. |
| `33_mean_picks_per_event.png` | Mean number of mapped picks among WAV+PICKS containers, grouped by waveform start date. Days without such containers are undefined and remain blank rather than becoming zero picks/event. |
| `33_primary_pick_fraction.png` | Primary picks divided by all mapped picks on each pick's own recorded date, including PICKS_ONLY records. A high fraction reflects source availability/priority, not confidence or accuracy. Days without picks are undefined. |

Daily count charts include every calendar day in the union of waveform and event coverage and use bars one day wide, without gaps between adjacent days. Dates are UTC days: steps 10, 20 and 21 converted waveform and pick times to UTC with each acquisition PC's rule in `clock_regimes.csv`; drift within a regime is not corrected. An event crossing midnight is counted on its start day; individual picks use their own dates.

The mean-pick computation joins explicitly on event ID rather than assigning dates by row position. `step30_event_csv` is retained in the settings for continuity but the plotted summaries use the step-32 catalogue and map. Outputs overwrite matching names in the external QC directory. The saved review is step 34.

## 1. Configuration

Inspect source paths, output paths and tolerances. This cell resolves configuration without running association or plotting.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    waveform_index=P['WAVEFORM_INDEX'],
    step30_event_csv=P['INDIV_WAVEFORM_EVENT_CSV'],
    event_catalog=P['WAVEFORM_PICK_EVENT_INDEX'],
    pick_map=P['PICK_MAP_CSV'],
    outdir=P['STEP33_DIR'],
)
display(SETTINGS)

## 2. Implementation

These definitions make the workflow inspectable here. Defining them does not write files; the run cell below executes them.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from lib.shared import to_dt, savefig

def run(*, waveform_index, step30_event_csv, event_catalog, pick_map, outdir):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    dtype = {k: str for k in ('event_id', 'waveform_id', 'pick_id')}
    wav = pd.read_csv(waveform_index, dtype=dtype, low_memory=False)
    ev = pd.read_csv(event_catalog, dtype=dtype, low_memory=False)
    picks = pd.read_csv(pick_map, dtype=dtype, low_memory=False)
    wav['date'] = to_dt(wav['starttime']).dt.floor('D')
    ev['date'] = to_dt(ev['starttime']).dt.floor('D')
    picks['date'] = to_dt(picks['pick_time']).dt.floor('D')
    bounds = pd.concat([wav['date'], ev['date']]).dropna()
    if bounds.empty:
        raise ValueError('No valid waveform/event dates to plot.')
    days = pd.date_range(bounds.min(), bounds.max(), freq='D', name='date')
    daily_wav = wav.groupby('date').size().reindex(days, fill_value=0).rename('n_waveforms')
    daily_ev = (ev.groupby(['date', 'event_class']).size().unstack(fill_value=0)
                .reindex(index=days, columns=['WAV+PICKS', 'WAV_ONLY', 'PICKS_ONLY'], fill_value=0))
    daily = pd.concat([daily_wav, daily_ev], axis=1)
    daily['waveform_count_difference'] = daily['n_waveforms'] - daily['WAV+PICKS'] - daily['WAV_ONLY']
    daily.to_csv(outdir / '33_daily_association_counts.csv')

    def bars(table, title, ylabel, filename):
        fig, ax = plt.subplots(figsize=(12, 4))
        bottom = pd.Series(0.0, index=table.index)
        for name in table.columns:
            ax.bar(table.index, table[name], bottom=bottom, width=1.0,
                   align='edge', linewidth=0, label=name)
            bottom = bottom + table[name].fillna(0)
        locator = mdates.AutoDateLocator(minticks=5, maxticks=12)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.set_xlim(days[0], days[-1] + pd.Timedelta(days=1))
        ax.set(title=title, ylabel=ylabel, xlabel='Date (UTC), 1991')
        ax.set_ylim(bottom=0)
        if 'fraction' in filename:
            ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(axis='y', alpha=0.2)
        ax.set_axisbelow(True)
        savefig(outdir / filename)

    bars(daily_ev[['WAV+PICKS', 'WAV_ONLY']], 'Waveform containers per day', 'Waveform files', '33_waveform_coverage.png')
    bars(daily_ev, 'Association containers per day (not unique earthquakes)', 'Containers', '33_event_types.png')
    counts = picks.groupby('event_id').size().rename('n_mapped_picks')
    linked = ev[ev['event_class'].eq('WAV+PICKS')][['event_id', 'date']].merge(
        counts, left_on='event_id', right_index=True, how='left', validate='one_to_one')
    daily_mean = linked.groupby('date')['n_mapped_picks'].mean().reindex(days)
    daily_mean.rename('mean_picks').to_csv(outdir / '33_mean_picks_per_event.csv')
    bars(daily_mean.rename('Mean picks').to_frame(), 'Mean picks per waveform container with picks', 'Mean picks/container', '33_mean_picks_per_event.png')
    pri = picks.groupby(['date', 'pick_priority']).size().unstack(fill_value=0).reindex(days, fill_value=0)
    numerator = pri['primary'] if 'primary' in pri else pd.Series(0, index=days)
    frac = numerator / pri.sum(axis=1).replace(0, float('nan'))
    frac.rename('primary_fraction').to_csv(outdir / '33_primary_pick_fraction.csv')
    bars(frac.rename('Primary fraction').to_frame(), 'Primary fraction of all mapped picks (including pick-only)', 'Fraction', '33_primary_pick_fraction.png')
    print('Step 33 complete:', outdir)
    print('Days with waveform count discrepancies:', daily['waveform_count_difference'].ne(0).sum())
    return dict(daily=daily, daily_mean=daily_mean, primary_fraction=frac, ev=ev, picks=picks)

## 3. Run

This cell reads the configured inputs and replaces configured outputs. Review the summary before continuing.

In [ ]:
result = run(**SETTINGS)

In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


In [ ]:
from IPython.display import Image

# Show saved diagnostics inline; all figures remain in the configured folder.
plot_roots = [Path(SETTINGS[k]) for k in ('outdir', 'out_dir', 'out')
              if k in SETTINGS and SETTINGS[k] is not None]
plots = sorted({p for root in plot_roots
                for p in ([root] if root.suffix.lower() == '.png'
                          else root.rglob('*.png')) if p.is_file()})
print(f'{len(plots)} saved PNG figures; showing up to four.')
for plot in plots[:4]:
    print(plot)
    display(Image(filename=str(plot)))
